In [1]:
import pandas as pd
import numpy as np
from src.features.process_race_header import preprocess_pipeline_race_header

# 0. Load race header dataset and trap bias statistics

In [ ]:
full_race_header = pd.read_csv("../data/intermediate/07_all_race_header.csv")
trap_bias_stats = pd.read_csv("../data/processed/05_trap_bias_stats.csv")

In [3]:
full_race_header.head()

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceType,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,raceForecast,raceTricast,meeting_Id,source_file
0,13:44:00,14/04/2018,1,NaN,13,Flat,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,(5-1) £26.94,(5-1-2) £60.36,337356,1.csv
1,11:07:00,14/04/2018,10,NaN,3,Flat,False,A3,480.0,1st £100 | Others £37 | Race Total £285,10.0,(1-4) £12.07,(1-4-6) £39.29,337356,10.csv
2,14:06:00,17/04/2019,100,NaN,1,Flat,False,A10,480.0,1st £105 | 2nd £40 | Others £40 | Race Total £305,-10.0,(1-2) £24.55,(1-2-3) £52.24,350084,100.csv
3,17:08:00,26/04/2019,1000,NaN,10,Flat,False,A8,380.0,1st £110 | 2nd £45 | Others £40 | Race Total £315,10.0,"(2-4) £23.87, (2-4) £23.87","(2-4-1) £36.97, (2-4-5) £34.51",350316,1000.csv
4,21:56:00,19/02/2014,10000,NaN,10,Flat,False,A1,400.0,1st £135 | Others £35 | Race Total £310,NaN,(3-1) £13.71,(3-1-5) £48.83,284014,10000.csv


# 1. Apply preliminary cleaning and processing

**TODO: add comments and def.**

In [4]:
race_header_cleaned = preprocess_pipeline_race_header(full_race_header)

In [5]:
race_header_cleaned.head()

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,...,meeting_Id,source_file,raceType_Flat,raceType_Hurdles,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,classGroup,qualityLevel
0,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,337356,1.csv,1,0,2018,4,5,104,graded_a,7
1,11:07:00,2018-04-14,10,NaN,3,False,A3,480.0,1st £100 | Others £37 | Race Total £285,10.0,...,337356,10.csv,1,0,2018,4,5,104,graded_a,11
2,14:06:00,2019-04-17,100,NaN,1,False,A10,480.0,1st £105 | 2nd £40 | Others £40 | Race Total £305,-10.0,...,350084,100.csv,1,0,2019,4,2,107,graded_a,4
3,17:08:00,2019-04-26,1000,NaN,10,False,A8,380.0,1st £110 | 2nd £45 | Others £40 | Race Total £315,10.0,...,350316,1000.csv,1,0,2019,4,4,116,graded_a,6
4,21:56:00,2014-02-19,10000,NaN,10,False,A1,400.0,1st £135 | Others £35 | Race Total £310,NaN,...,284014,10000.csv,1,0,2014,2,2,50,graded_a,13


# 2. Go fetch track names

An issue with the dataset is that the track names are missing, which is an important information to have, mainly to associate the trap bias.

So we have to go fetch it on the website where we got the data in the first place.

In [6]:
import requests
# ---------------- API ----------------

def fetch_track_name(meeting_id: str):
    url_test = f"https://api.gbgb.org.uk/api/results/meeting/{meeting_id}?meeting={meeting_id}"
    response = requests.get(url=url_test, timeout=15)
    response.raise_for_status()
    r = response.json()
    response.close()

    if not r or not isinstance(r, list) or len(r) == 0:
        raise ValueError("Empty response for meeting_id")

    track_name = r[0].get("trackName")
    return track_name

In [7]:
meeting_col = "meeting_Id"
meeting_ids = race_header_cleaned[meeting_col].astype(str).unique().tolist()
meeting_id_to_track = {}

for meeting_id in meeting_ids:
    if meeting_id in meeting_id_to_track:
        continue
    try:
        meeting_id_to_track[meeting_id] = fetch_track_name(meeting_id)
    except Exception:
        meeting_id_to_track[meeting_id] = None

race_header_cleaned["trackName"] = race_header_cleaned[meeting_col].astype(str).map(meeting_id_to_track)

# 3. Trap bias per track/distance 

In [8]:
trap_lookup = (
    trap_bias_stats
    .set_index(["trackName", "raceDistance", "trapNumber"])["trapBiasResultPosition"]
    .to_dict()
)

for t in range(1, 7):
    race_header_cleaned[f"trap{t}Bias"] = race_header_cleaned.apply(
        lambda row, t=t: trap_lookup.get((row["trackName"], row["raceDistance"], t), np.nan),
        axis=1
    )

In [9]:
race_header_cleaned

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,...,raceDayOfYear,classGroup,qualityLevel,trackName,trap1Bias,trap2Bias,trap3Bias,trap4Bias,trap5Bias,trap6Bias
0,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
1,11:07:00,2018-04-14,10,NaN,3,False,A3,480.0,1st £100 | Others £37 | Race Total £285,10.0,...,104,graded_a,11,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
2,14:06:00,2019-04-17,100,NaN,1,False,A10,480.0,1st £105 | 2nd £40 | Others £40 | Race Total £305,-10.0,...,107,graded_a,4,Monmore,3.0,3.0,3.0,4.0,4.0,3.0
3,17:08:00,2019-04-26,1000,NaN,10,False,A8,380.0,1st £110 | 2nd £45 | Others £40 | Race Total £315,10.0,...,116,graded_a,6,Crayford,3.0,3.0,3.0,4.0,4.0,3.0
4,21:56:00,2014-02-19,10000,NaN,10,False,A1,400.0,1st £135 | Others £35 | Race Total £310,NaN,...,50,graded_a,13,Romford,3.0,3.0,4.0,4.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540445,20:12:00,2023-12-27,999979,Christmas Owners' Bonus Series 18th Trophy,8,False,D2,277.0,1st £180 | Others £60 Race Total £480,-5.0,...,361,distance,4,Central Park,3.0,3.0,3.0,4.0,4.0,3.0
540446,20:27:00,2023-12-27,999980,Christmas Owners' Bonus Series 19th Trophy,9,False,A4,491.0,1st £195 | Others £65 Race Total £520,-10.0,...,361,graded_a,10,Central Park,3.0,3.0,3.0,3.0,4.0,3.0
540447,20:46:00,2023-12-27,999981,Christmas Owners' Bonus Series 20th Trophy,10,False,D3,277.0,1st £155 | Others £55 Race Total £430,-5.0,...,361,distance,3,Central Park,3.0,3.0,3.0,4.0,4.0,3.0
540448,21:01:00,2023-12-27,999982,Christmas Owners' Bonus Series 21st Trophy,11,False,A3,491.0,1st £205 | Others £65 Race Total £530,-10.0,...,361,graded_a,11,Central Park,3.0,3.0,3.0,3.0,4.0,3.0


# 4. Final cleaning

## 4.1 Keep the usefull columns and fill by mean

In [ ]:
cols_to_remove = ["raceTime", "raceTitle", "raceNumber", "raceHandicap", 
                  "raceClass", "racePrizes", "raceGoing", "raceForecast", "raceTricast", 
                  "raceType_Flat", "raceType_Hurdles"]
cols_to_keep = [col for col in race_header_cleaned.columns if col not in cols_to_remove]

In [20]:
final_race_header = race_header_cleaned[cols_to_keep].copy()
numeric_cols = final_race_header.select_dtypes(include=[np.number]).columns
mean_val = final_race_header[numeric_cols].mean(skipna=True)    # mean without outliers
final_race_header.fillna(mean_val, inplace=True)

,raceId,raceDistance,meeting_Id,source_file,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,classGroup,qualityLevel,trackName,trap1Bias,trap2Bias,trap3Bias,trap4Bias,trap5Bias,trap6Bias
0,1,480.0,337356,1.csv,2018,4,5,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
1,10,480.0,337356,10.csv,2018,4,5,104,graded_a,11,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
2,100,480.0,350084,100.csv,2019,4,2,107,graded_a,4,Monmore,3.0,3.0,3.0,4.0,4.0,3.0
3,1000,380.0,350316,1000.csv,2019,4,4,116,graded_a,6,Crayford,3.0,3.0,3.0,4.0,4.0,3.0
4,10000,400.0,284014,10000.csv,2014,2,2,50,graded_a,13,Romford,3.0,3.0,4.0,4.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540445,999979,277.0,405083,999979.csv,2023,12,2,361,distance,4,Central Park,3.0,3.0,3.0,4.0,4.0,3.0
540446,999980,491.0,405083,999980.csv,2023,12,2,361,graded_a,10,Central Park,3.0,3.0,3.0,3.0,4.0,3.0
540447,999981,277.0,405083,999981.csv,2023,12,2,361,distance,3,Central Park,3.0,3.0,3.0,4.0,4.0,3.0
540448,999982,491.0,405083,999982.csv,2023,12,2,361,graded_a,11,Central Park,3.0,3.0,3.0,3.0,4.0,3.0


In [21]:
final_race_header.describe()

,raceId,raceDistance,meeting_Id,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,qualityLevel,trap1Bias,trap2Bias,trap3Bias,trap4Bias,trap5Bias,trap6Bias
count,5.404500e+05,540450.000000,540450.000000,540450.000000,540450.000000,540450.000000,540450.000000,540450.00000,540450.000000,540450.000000,540450.000000,540450.000000,540450.000000,540450.000000
mean,5.292611e+05,446.367710,350204.988617,2018.913526,6.433813,3.178377,180.365960,8.28994,3.004544,3.062157,3.163677,3.693861,3.945037,3.158981
std,3.286558e+05,84.301614,38682.661157,3.152281,3.482064,1.921680,106.344668,3.90564,0.073835,0.242378,0.370543,0.461475,0.228984,0.366809
min,1.000000e+00,0.000000,278770.000000,2013.000000,1.000000,0.000000,1.000000,0.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.426022e+05,415.000000,317525.000000,2016.000000,3.000000,2.000000,85.000000,6.00000,3.000000,3.000000,3.000000,3.000000,4.000000,3.000000
50%,5.188495e+05,470.000000,349341.000000,2019.000000,6.000000,3.000000,180.000000,9.00000,3.000000,3.000000,3.000000,4.000000,4.000000,3.000000
75%,8.120668e+05,480.000000,381765.000000,2022.000000,9.000000,5.000000,274.000000,11.00000,3.000000,3.000000,3.000000,4.000000,4.000000,3.000000
max,1.119840e+06,1048.000000,427752.000000,2025.000000,12.000000,6.000000,366.000000,16.00000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000


## 4.2 Keep only A graded races and all the races

In [23]:
Aclass_race_header = final_race_header[final_race_header["classGroup"] == "graded_a"].copy()

In [ ]:
final_race_header.to_csv("../data/processed/09_all_race_header.csv", index=False)
Aclass_race_header.to_csv("../data/processed/09_graded_a_race_header.csv", index=False)